In [1]:
import pandas as pd
import numpy as np

In [2]:
!pip install transformers datasets peft accelerate bitsandbytes


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset
from peft import PeftModel
import torch


In [4]:
# Load tokenizer and base model (already LoRA-adapted)
tokenizer = AutoTokenizer.from_pretrained("final_model_tokenizer/lora_mentor_model_tokenizer")
model = AutoModelForCausalLM.from_pretrained("final_model_tokenizer/lora_mentor_model", device_map="auto", torch_dtype=torch.float16)

In [5]:
df = pd.read_json("datasets/human_dataset.json")

In [6]:
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

In [7]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [8]:
def format_prompt(example):
    return {
        "text": f"<|user|>\n{example['prompt']}\n<|assistant|>\n{example['response']}"
    }

In [9]:
train_dataset = train_dataset.map(format_prompt)
test_dataset = test_dataset.map(format_prompt)

Map:   0%|          | 0/27032 [00:00<?, ? examples/s]

Map:   0%|          | 0/3004 [00:00<?, ? examples/s]

In [10]:
def tokenize(example):
    enc = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    enc["labels"] = enc["input_ids"].copy()  # For causal LM, labels are the same as input_ids
    return enc

In [11]:
train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=train_dataset.column_names)
test_dataset = test_dataset.map(tokenize, batched=True, remove_columns=test_dataset.column_names)

Map:   0%|          | 0/27032 [00:00<?, ? examples/s]

Map:   0%|          | 0/3004 [00:00<?, ? examples/s]

In [12]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [14]:
training_args = TrainingArguments(
    output_dir="./lora_mentor_model_v2",  # Save path for the fine-tuned model
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-5,
    fp16=True,
    logging_steps=10,
    save_steps=500,  # Optionally save the model every 500 steps
    save_total_limit=2,  # Optionally keep only the last 2 saved checkpoints
    report_to="none"
)

In [15]:
# Step 8: Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

In [19]:
# Step 1: Unfreeze model parameters if frozen
for param in model.parameters():
    param.requires_grad = True



# Step 3: Check if gradients are being computed
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Training parameter: {name}")

Training parameter: model.embed_tokens.weight
Training parameter: model.layers.0.self_attn.q_proj.base_layer.weight
Training parameter: model.layers.0.self_attn.q_proj.lora_A.default.weight
Training parameter: model.layers.0.self_attn.q_proj.lora_B.default.weight
Training parameter: model.layers.0.self_attn.k_proj.weight
Training parameter: model.layers.0.self_attn.v_proj.base_layer.weight
Training parameter: model.layers.0.self_attn.v_proj.lora_A.default.weight
Training parameter: model.layers.0.self_attn.v_proj.lora_B.default.weight
Training parameter: model.layers.0.self_attn.o_proj.weight
Training parameter: model.layers.0.mlp.gate_proj.weight
Training parameter: model.layers.0.mlp.up_proj.weight
Training parameter: model.layers.0.mlp.down_proj.weight
Training parameter: model.layers.0.input_layernorm.weight
Training parameter: model.layers.0.post_attention_layernorm.weight
Training parameter: model.layers.1.self_attn.q_proj.base_layer.weight
Training parameter: model.layers.1.self

In [20]:
# Step 9: Start training
trainer.train()

AssertionError: No inf checks were recorded for this optimizer.

In [ ]:
# Step 10: Save the fine-tuned model and tokenizer
model.save_pretrained("lora_mentor_model_v2")
tokenizer.save_pretrained("lora_mentor_model_tokenizer_v2")